# Step 9 — Reporter

Packages what Steps 6–8 already produced. No new mechanisms, no new economic logic — `src/engine.py`, `src/policy_model.py` and `src/portfolio_sweep.py` are **consumed, not modified**.

Three findings this step makes visible to a reader outside this project:

- **D-062** — the four outputs are reported as **avoidable** cost (conversion excludes €3.6M/year fixed absorption no lever moves), with the fixed figure shown separately and labelled, never dropped
- **D-066** — the per-SKU policy optimum (Step 7) is over cover and service only; `min_run_hours` is a line decision
- **D-070** — a warning on the specific lever region Step 8 found breaches capacity, built to generalise rather than hard-code today's one finding (L2, high cover)

Prerequisite: Steps 4, 5a, 7 and 8 have already been run and their outputs exist. This notebook checks for each and names the producing step if missing — the reporter never recomputes an upstream artefact.

## Setup

In [ ]:
import subprocess, os, sys
def sh(cmd, cwd=None):
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    print(r.stdout[-3000:])
    if r.returncode != 0: print("STDERR:", r.stderr[-1500:])
    return r
REPO = '/content/ibp-tradeoff'
os.chdir('/content'); sh('rm -rf ibp-tradeoff')
sh('git clone https://github.com/rdelolmog-creator/ibp-tradeoff.git')
os.chdir(REPO); sys.path.insert(0, REPO)
print('cwd:', os.getcwd())

## Check upstream artefacts

Steps 4, 5a, 7 and 8 must already have been run — this step does not recompute them. `require_artefact` names which notebook produces a file if it's missing.

In [ ]:
import pandas as pd, numpy as np, yaml
from src.reporter import require_artefact, ReporterViolation

NEEDED = [
    'data_primary/clean/clean_master.parquet',
    'data_primary/clean/sku_master.parquet',
    'data_primary/clean/demand_characteristics.csv',
    'data_primary/clean/censoring_diagnostics.csv',
]
missing = []
for f in NEEDED:
    try:
        require_artefact(f)
        print('OK  ', f)
    except ReporterViolation as e:
        print('MISSING', f); missing.append(str(e))

if missing:
    print()
    for m in missing: print(m)
    print()
    print('Upload optimal_policy.csv, portfolio_sweep.csv, step08_portfolio_summary.csv,')
    print('and step08_lever_consistency.csv from your Step 7/8 Drive outputs before continuing.')

## Rebuild Step 4, upload Step 5a, then upload Step 7/8 outputs

Same pattern as Steps 6–8. The four Step 7/8 CSVs come from `My Drive/ibp-tradeoff-outputs/` if you ran those notebooks — upload them here.

In [ ]:
from src.ingest import DataIngestor
from src.cleaner import DataCleaner
if not os.path.isdir('data_primary/raw'):
    sh('python generate_data.py'); sh('mv data data_primary')
ing = DataIngestor(repo_root=REPO, data_root='data_primary')
clean_master, sku_master, _ = DataCleaner(ing.schema, ing.assumptions).clean(ing.load())
os.makedirs('data_primary/clean', exist_ok=True)
clean_master.to_parquet('data_primary/clean/clean_master.parquet', index=False)
sku_master.to_parquet('data_primary/clean/sku_master.parquet', index=False)
print('clean_master:', clean_master.shape, '| sku_master:', sku_master.shape)

In [ ]:
from google.colab import files
import shutil
print('Select demand_characteristics.csv AND censoring_diagnostics.csv (Step 5a):')
uploaded = files.upload()
demand_characteristics = pd.read_csv('demand_characteristics.csv').set_index('sku_id')
censoring_diagnostics  = pd.read_csv('censoring_diagnostics.csv').set_index('sku_id')
from src.portfolio_impact import get_flagged_skus
flagged = get_flagged_skus(censoring_diagnostics.reset_index())
for f in ('demand_characteristics.csv', 'censoring_diagnostics.csv'):
    shutil.copy(f, f'data_primary/clean/{f}')
print(f'flagged: {len(flagged)}')

In [ ]:
print('Select optimal_policy.csv, portfolio_sweep.csv, step08_portfolio_summary.csv, '
      'step08_lever_consistency.csv (Step 7 + Step 8 outputs):')
uploaded2 = files.upload()
optimal_policy          = pd.read_csv('optimal_policy.csv')
portfolio_sweep         = pd.read_csv('portfolio_sweep.csv')
portfolio_summary       = pd.read_csv('step08_portfolio_summary.csv')
lever_consistency_table = pd.read_csv('step08_lever_consistency.csv')
print('optimal_policy:', optimal_policy.shape, '| portfolio_summary:', portfolio_summary.shape)

## Build the engine

In [ ]:
from src.engine import TradeOffEngine, LeverSettings, build_line_master
assumptions = yaml.safe_load(open('config/assumptions.yaml'))
schema      = yaml.safe_load(open('config/schema.yaml'))
engine = TradeOffEngine(assumptions, schema, clean_master, sku_master,
                        demand_characteristics, flagged)
line_master = build_line_master(assumptions, schema)
print('assumption fingerprint:', engine.assumption_fingerprint)

## Avoidable cost view — the Step 6 gate scenario, D-062

Both `total_avoidable_cost_eur` and `total_reported_cost_eur` are always returned together — reporting only the avoidable figure without the reconciling total is exactly how a plant controller ends up thinking the model lost €3.6M of their P&L.

In [ ]:
from src.reporter import (avoidable_cost_view, owner_view, capacity_warning,
                          policy_brief, portfolio_brief, export_artefacts)

MVD_LINE = 'L3'
cat = str(engine.sku_master.loc[engine.line_skus(MVD_LINE)[0], 'category'])
base = LeverSettings.defaults(assumptions, cat, 'A')
gate_scenario = engine.run_scenario(MVD_LINE, base)

avoidable = avoidable_cost_view(gate_scenario, assumptions, schema)
print(f'{"":30}{"EUR":>16}')
for k in ('lost_sales_eur','excess_obsolescence_eur','working_capital_cost_eur',
          'conversion_cost_avoidable_eur','conversion_cost_fixed_eur'):
    print(f'{k:<30}{avoidable[k]:>16,.0f}')
print('-'*46)
print(f'{"total_avoidable_cost_eur":<30}{avoidable["total_avoidable_cost_eur"]:>16,.0f}')
print(f'{"total_reported_cost_eur":<30}{avoidable["total_reported_cost_eur"]:>16,.0f}   <- reconciles to the full P&L figure')

## Owner view — accountability at role level (architecture §11)

In [ ]:
owners = owner_view(avoidable, assumptions)
print(owners.to_string(index=False))

## Capacity warning — D-070, generalised

Reads each line's own worst-case condition from `portfolio_summary` — does not hard-code "L2" or a cover value, so it fires on whatever line/setting develops the same pattern later.

In [ ]:
ps_indexed = portfolio_summary.set_index('line_id')
tests = [('L2', 10.0), ('L2', 4.0), ('L3', 10.0)]
for line_id, cover in tests:
    msg = capacity_warning(line_id, cover, ps_indexed.loc[line_id])
    print(f'{line_id} @ cover={cover}: {msg or "(no warning)"}')

## Policy brief — Step 7's optimum, made actionable

In [ ]:
default_levers_by_class = {
    c: {'inventory_cover_weeks': assumptions['abc'][c]['target_cover_weeks'],
        'service_target': assumptions['abc'][c]['service_floor']}
    for c in ('A', 'B', 'C')
}
brief_table = policy_brief(optimal_policy, sku_master, default_levers_by_class, top_n=10)
print(brief_table.to_string(index=False))

## Portfolio brief — the paragraph a CFO reads first

In [ ]:
brief_text = portfolio_brief(portfolio_summary, lever_consistency_table)
print(brief_text)

## Export the three artefacts

In [ ]:
paths = export_artefacts(avoidable, brief_table, brief_text, out_dir='.')
print(paths)

## Tests

In [ ]:
sh('python -m pytest tests/test_reporter.py -q --no-header')
sh('python -m pytest tests/test_engine.py tests/test_policy_model.py '
  'tests/test_portfolio_sweep.py tests/test_pipeline.py -q --no-header')

## Consolidated report — the only cell to copy

In [ ]:
import hashlib, subprocess

t_rep  = subprocess.run('python -m pytest tests/test_reporter.py -q --no-header',
                        shell=True, capture_output=True, text=True)
t_eng  = subprocess.run('python -m pytest tests/test_engine.py -q --no-header',
                        shell=True, capture_output=True, text=True)
t_pol  = subprocess.run('python -m pytest tests/test_policy_model.py -q --no-header',
                        shell=True, capture_output=True, text=True)
t_port = subprocess.run('python -m pytest tests/test_portfolio_sweep.py -q --no-header',
                        shell=True, capture_output=True, text=True)
if not os.path.isdir('data_control/raw'):
    subprocess.run('cp config/assumptions.yaml config/_bk.yaml && '
                   'cp config/assumptions_lowcensoring.yaml config/assumptions.yaml && '
                   'python generate_data.py && '
                   'cp config/_bk.yaml config/assumptions.yaml && rm config/_bk.yaml && '
                   'mv data data_control', shell=True, capture_output=True, text=True)
t_pipe = subprocess.run('python -m pytest tests/test_pipeline.py -q --no-header',
                        shell=True, capture_output=True, text=True)

L=[]; w=L.append
w('='*78); w('STEP 9 - REPORTER - CONSOLIDATED REPORT'); w('='*78)
w(f'assumption set   : {engine.assumption_fingerprint}')
w(f'engine.py sha    : {hashlib.sha256(open("src/engine.py","rb").read()).hexdigest()[:12]}')
w(f'reporter.py sha  : {hashlib.sha256(open("src/reporter.py","rb").read()).hexdigest()[:12]}')
w(f'pandas {pd.__version__} / numpy {np.__version__}')

w(''); w('-- 1. AVOIDABLE COST VIEW - Step 6 gate scenario '+'-'*28)
for k in ('lost_sales_eur','excess_obsolescence_eur','working_capital_cost_eur',
          'conversion_cost_avoidable_eur','conversion_cost_fixed_eur',
          'total_avoidable_cost_eur','total_reported_cost_eur'):
    w(f'{k:<32}{avoidable[k]:>16,.0f}')

w(''); w('-- 2. OWNER VIEW '+'-'*61)
w(owners.to_string(index=False))

w(''); w('-- 3. CAPACITY WARNING (D-070) '+'-'*47)
for line_id, cover in tests:
    msg = capacity_warning(line_id, cover, ps_indexed.loc[line_id])
    w(f'{line_id} @ cover={cover}: {msg or "(no warning)"}')

w(''); w('-- 4. POLICY BRIEF (top 10 by saving) '+'-'*40)
w(brief_table.round(3).to_string(index=False))

w(''); w('-- 5. PORTFOLIO BRIEF '+'-'*56)
w(brief_text)

w(''); w('-- 6. TESTS '+'-'*66)
for label, r in (('test_reporter.py', t_rep), ('test_engine.py', t_eng),
                 ('test_policy_model.py', t_pol), ('test_portfolio_sweep.py', t_port),
                 ('test_pipeline.py', t_pipe)):
    w(f'{label:<24}: ' + (r.stdout.strip().splitlines() or ['no output'])[-1])
if any(r.returncode for r in (t_rep,t_eng,t_pol,t_port,t_pipe)):
    w(''); w('FAILURES:')
    for r in (t_rep,t_eng,t_pol,t_port,t_pipe):
        if r.returncode: w(r.stdout[-2500:])

w(''); w('-- 7. CHECKS '+'-'*65)
checks = [
 ('both avoidable and reported totals present',
  'total_avoidable_cost_eur' in avoidable and 'total_reported_cost_eur' in avoidable),
 ('reported total exceeds avoidable total (fixed cost adds back)',
  avoidable['total_reported_cost_eur'] > avoidable['total_avoidable_cost_eur']),
 ('capacity warning fires on the known breach condition',
  capacity_warning('L2', 10.0, ps_indexed.loc['L2']) is not None),
 ("capacity warning silent at that line's own optimum",
  capacity_warning('L2', 4.0, ps_indexed.loc['L2']) is None),
 ('owner mapping present and used', len(owners) > 0),
 ('all test suites pass', all(r.returncode==0 for r in (t_rep,t_eng,t_pol,t_port,t_pipe))),
]
for label, ok in checks:
    w(f'  [{"PASS" if ok else "SEE NOTE"}]  {label}')
w('')
w('Magnitudes are not findings (arch section 10). Every number above is')
w('whatever the generator and the assumption set encoded.')
w('='*78)

report_text = '\n'.join(L)
open('step09_report.txt','w').write(report_text)
try:
    import shutil
    d = '/content/drive/My Drive/ibp-tradeoff-outputs'
    if os.path.isdir(d):
        for f in ('step09_report.txt','avoidable_cost_summary.csv',
                  'policy_brief.csv','portfolio_brief.txt'):
            if os.path.exists(f): shutil.copy(f, d)
        print('saved to', d, '\\n')
except Exception as e:
    print('Drive copy skipped:', e, '\\n')
print(report_text)